# 📊 Multi-Model Independent Evaluator & Leaderboard

This interactive dashboard allows you to define multiple YOLO models (standard or SAHI) and run an independent evaluation across a shared Test Dataset. Features:
- **Interactive Configuration**: Add dynamically N models, their paths, and SAHI configurations.
- **Quantitative Leaderboard**: Calculate metrics (Precision, Recall, F1-Score, mAP50) natively and sort the results.
- **Visual Comparison**: Upload a 4K raw image and view `1xN` subplots displaying their predictive differences side-by-side.

In [9]:
# Install libraries if not available
!pip install -q ultralytics ipywidgets shapely opencv-python matplotlib pandas sahi

import os
import sys
from pathlib import Path
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import pyplot as plt
import pandas as pd
from ultralytics import YOLO

# Add root directory to python path to import our custom pipeline
sys.path.append(str(Path(os.path.abspath('')).parent.parent))
try:
    from src.inference import MaskPostProcessor, RockVisualizer
except ImportError:
    print("Warning: src.inference not found. Ensure you are running this from the correct root!")

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## ⚙️ 1. Interactive Model Configuration

Define your test dataset `data.yaml` path, choose the number of models to compare, and input their parameters.
Check the **"Use SAHI"** box if the respective model should use slicing inference.

In [10]:
# -------------------- UI Configuration --------------------
style = {'description_width': 'initial'}
layout = widgets.Layout(width='500px')

# Global storage mapping
gui_config = {
    "dataset_yaml": "",
    "models": []
}

default_yaml = str(Path.home() / "projects/DwiAnggara/Datasets/Batch3and4_YOLO_SAHI/data.yaml")
if not os.path.exists(default_yaml):
    default_yaml = ""

dataset_widget = widgets.Text(
    value=default_yaml,
    description='Dataset YAML Path:',
    style=style, layout=widgets.Layout(width='700px')
)

num_models_widget = widgets.BoundedIntText(
    value=2, min=1, max=10, step=1,
    description='Number of Models:',
    style=style, layout=widgets.Layout(width='200px')
)

generate_btn = widgets.Button(description='Generate Inputs', button_style='info')
save_config_btn = widgets.Button(description='Save Config', button_style='success', icon='save')
config_output = widgets.Output()

models_container = widgets.VBox([])
model_widgets_mapping = []

def generate_model_inputs(b):
    models_container.children = []
    model_widgets_mapping.clear()
    
    rows = []
    for i in range(num_models_widget.value):
        row_title = widgets.HTML(f"<b>Model {i+1}</b>")
        name_input = widgets.Text(value=f"YOLO_M{i+1}", description="Name:", layout=widgets.Layout(width='200px'))
        path_input = widgets.Text(value="", placeholder="Paste best.pt absolute path...", description="Path:", layout=widgets.Layout(width='450px'))
        sahi_check = widgets.Checkbox(value=False, description="Use SAHI", layout=widgets.Layout(width='150px'))
        
        row_box = widgets.HBox([name_input, path_input, sahi_check])
        rows.append(widgets.VBox([row_title, row_box]))
        
        model_widgets_mapping.append({
            "name": name_input,
            "path": path_input,
            "use_sahi": sahi_check
        })
        
    models_container.children = rows

def save_configuration(b):
    with config_output:
        clear_output()
        dataset_yaml = dataset_widget.value
        
        if not os.path.exists(dataset_yaml):
            print(f"❌ Error: Dataset file not found at {dataset_yaml}")
            return
            
        gui_config['dataset_yaml'] = dataset_yaml
        gui_config['models'] = []
        
        valid = True
        for idx, w in enumerate(model_widgets_mapping):
            m_name = w['name'].value
            m_path = w['path'].value
            m_sahi = w['use_sahi'].value
            
            if not os.path.exists(m_path):
                print(f"❌ Error: Weights for '{m_name}' not found at {m_path}")
                valid = False
            else:
                gui_config['models'].append({
                    "name": m_name,
                    "path": m_path,
                    "use_sahi": m_sahi
                })
        
        if valid:
            print("✅ Configuration saved successfully!")
            print(f"📂 Dataset: {dataset_yaml}")
            for m in gui_config['models']:
                sahi_flag = "SAHI" if m['use_sahi'] else "Standard"
                print(f"  🤖 [{sahi_flag}] {m['name']} -> {os.path.basename(m['path'])}")

generate_btn.on_click(generate_model_inputs)
save_config_btn.on_click(save_configuration)

# Initialize defaults
generate_model_inputs(None)

display(widgets.VBox([
    dataset_widget,
    widgets.HBox([num_models_widget, generate_btn]),
    widgets.HTML("<hr>"),
    models_container,
    widgets.HTML("<br>"),
    save_config_btn,
    config_output
]))

## 📈 2. Quantitative Leaderboard Generation

Once your configuration is saved above, run the cell below to extract native evaluation metrics for each model against the test dataset. The loop collects Precision, Recall, Validation mAP, masks IoU, and F1-Scores to rank them. SAHI quantitative logic defaults to native sliced metrics on pre-sliced `test` data.

In [ ]:
import logging
# Suppress Ultralytics verbose validation logs like "corrupt JPEG restored and saved"
logging.getLogger("ultralytics").setLevel(logging.ERROR)

evaluate_btn = widgets.Button(description='Run Benchmark Evaluator', button_style='primary', icon='rocket')
eval_output = widgets.Output()

def run_benchmarks(b):
    with eval_output:
        clear_output(wait=True)
        if not gui_config['models']:
            print("⚠️ Please configure and save your models first.")
            return
            
        dataset_yaml = gui_config['dataset_yaml']
        results_list = []
        
        for cfg in gui_config['models']:
            print(f"\n⏳ Starting evaluation for: {cfg['name']} {'(SAHI Mode)' if cfg['use_sahi'] else '(Standard Mode)'}")
            
            try:
                # Load model silently
                model = YOLO(cfg['path'])
                
                # We use native validation 
                print(f"   🚀 Validating {cfg['name']} natively against test split...")
                # Temporarily disable printing so Jupyter UI isn't flooded
                results = model.val(data=dataset_yaml, split='test', conf=0.25, verbose=False)
                
                # Extraction
                if hasattr(results, 'seg') and results.seg is not None:
                    p = float(results.seg.mp)
                    r = float(results.seg.mr)
                    map50 = float(results.seg.map50)
                    map50_95 = float(results.seg.map)
                else:
                    p = float(results.box.mp) if hasattr(results, 'box') else 0.0
                    r = float(results.box.mr) if hasattr(results, 'box') else 0.0
                    map50 = float(results.box.map50) if hasattr(results, 'box') else 0.0
                    map50_95 = float(results.box.map) if hasattr(results, 'box') else 0.0
                
                # Latency calculations
                latency = 0.0
                fps = 0.0
                if hasattr(results, 'speed'):
                    latency = sum(results.speed.values())
                    fps = 1000.0 / latency if latency > 0 else 0.0
                    
                f1_score = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
                
                # Append formatted results
                results_list.append({
                    "Model Name": cfg['name'],
                    "Type": "SAHI" if cfg['use_sahi'] else "Standard",
                    "F1-Score": f"{f1_score * 100:.2f}%",
                    "Precision": f"{p * 100:.2f}%",
                    "Recall": f"{r * 100:.2f}%",
                    "mAP@50": f"{map50 * 100:.2f}%",
                    "mAP@50-95": f"{map50_95 * 100:.2f}%",
                    "Avg IoU": "Not Supported", # YOLO val doesn't compute a single global avg IoU natively
                    "Avg Latency": f"{latency:.1f}ms",
                    "FPS": f"{fps:.1f}",
                    "_f1_raw": f1_score # For sorting
                })
                print(f"   ✅ Done.")
                
            except Exception as e:
                print(f"   ❌ Error evaluating {cfg['name']}: {e}")
                
        # Build pandas DataFrame for Leaderboard
        if results_list:
            df = pd.DataFrame(results_list)
            # Sort by F1-Score raw
            df = df.sort_values(by="_f1_raw", ascending=False).reset_index(drop=True)
            # Assign ranks
            df.insert(0, "Rank", df.index + 1)
            # Drop raw column
            df = df.drop(columns=["_f1_raw"])
            
            print("\n🏆 == MULTI-MODEL LEADERBOARD ==")
            display(df)
        
evaluate_btn.on_click(run_benchmarks)
display(evaluate_btn, eval_output)

Button(button_style='primary', description='Run Benchmark Evaluator', icon='rocket', style=ButtonStyle())

Output()

### ℹ️ Evaluation Notes & FAQ:
* **F1-Score**: Harmonic mean of Precision and Recall. Useful when class distribution is uneven.
* **mAP@50**: Mean Average Precision evaluated comprehensively at a `0.50` Intersection over Union (IoU).
* **mAP@50-95**: The average of mAP comprehensively calculated at multiple IoU thresholds (from `0.50` to `0.95` with a step size of `0.05`). 
* **Why Avg IoU is "Not Supported"**: YOLO evaluates datasets natively holistically utilizing `mAP` metrics across the `0.50` to `0.95` IoU scales rather than calculating a raw global scalar scalar instance-wise mean IoU.
* **Corrupted JPEGs**: If there are `corrupt JPEG restored and saved` warnings from stdout or standard loggers, it means Python Image Libraries (`PIL` or OpenCV) successfully resolved and stripped minor byte-level stream anomalies inside the raw validation input images. Ultralytics' `logging` module level has been securely constrained dynamically in the loop to sidestep this unwanted UI console inundation gracefully.

## 👁️ 3. Visual Multi-Model Comparison

Upload a raw 4K image to observe the prediction differences among all configured models visually side-by-side. The hollow polygon output mirrors CVAT annotation styles and utilizes exact coordinate projections.

In [ ]:
upload_compare_btn = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Test Image', style=style, layout=widgets.Layout(width='300px'))
conf_slider = widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05, description='Confidence:', style=style)
min_area_slider = widgets.IntSlider(value=300, min=0, max=5000, step=50, description='Min Area (px²):', style=style)
epsilon_slider = widgets.FloatSlider(value=0.01, min=0.001, max=0.05, step=0.001, readout_format='.3f', description='Epsilon Ratio:', style=style)

visualize_btn = widgets.Button(description='Compare visually', button_style='warning', icon='eye')
compare_output = widgets.Output()

def run_multi_visualizer(b):
    with compare_output:
        clear_output(wait=True)
        if not gui_config['models']:
            print("⚠️ Please configure and save your models first.")
            return
            
        if not upload_compare_btn.value:
            print("⚠️ Please upload an image first.")
            return
            
        print("⏳ Processing visual comparisons across all models...")
        
        # Read image
        uploaded_file = list(upload_compare_btn.value.values())[0] if isinstance(upload_compare_btn.value, dict) else upload_compare_btn.value[0]
        content = uploaded_file['content']
        nparr = np.frombuffer(content, np.uint8)
        original_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        num_models = len(gui_config['models'])
        
        # Dynamic Matplotlib grid: 1 Row, N Columns tightly packed
        h, w = original_bgr.shape[:2]
        # Match figsize to image aspect ratio to avoid white space above/below images
        fig, axes = plt.subplots(1, num_models, figsize=(8 * num_models, 8 * h / w))
        fig.subplots_adjust(wspace=0.02, hspace=0)
        
        if num_models == 1:
            axes = [axes] # Wrap to list for iteration
            
        post_processor = MaskPostProcessor(min_area=min_area_slider.value, epsilon_ratio=epsilon_slider.value, simplify_tol=2.0)
        visualizer = RockVisualizer(thickness=8)
        
        for ax, cfg in zip(axes, gui_config['models']):
            proc_polygons_by_class = {}
            print(f"   -> Masking {cfg['name']}...")
            
            try:
                if cfg['use_sahi']:
                    from sahi import AutoDetectionModel
                    from sahi.predict import get_sliced_prediction
                    import torch
                    
                    # Handle CPU fallback seamlessly
                    device_mode = "cuda:0" if torch.cuda.is_available() else "cpu"
                    
                    sahi_model = AutoDetectionModel.from_pretrained(
                        model_type="yolov8",
                        model_path=cfg['path'],
                        confidence_threshold=conf_slider.value,
                        device=device_mode,
                    )
                    
                    result = get_sliced_prediction(
                        original_bgr,
                        sahi_model,
                        slice_height=640,
                        slice_width=640,
                        overlap_height_ratio=0.2,
                        overlap_width_ratio=0.2,
                        verbose=False
                    )
                    
                    for obj in result.object_prediction_list:
                        if obj.mask is not None:
                            clean_polys = post_processor.process(obj.mask.bool_mask)
                            idx = obj.category.id
                            if idx not in proc_polygons_by_class:
                                proc_polygons_by_class[idx] = []
                            proc_polygons_by_class[idx].extend(clean_polys)
                else:
                    yolo_model = YOLO(cfg['path'])
                    results = yolo_model.predict(original_bgr, conf=conf_slider.value, verbose=False)[0]
                    
                    if results.masks is not None:
                        mask_xys = results.masks.xy
                        class_ids = results.boxes.cls.cpu().numpy()
                        h_shape, w_shape = original_bgr.shape[:2]
                        
                        for mask_xy, cls_id in zip(mask_xys, class_ids):
                            idx = int(cls_id)
                            if len(mask_xy) > 2:
                                mask_xy_int = mask_xy.astype(np.int32)
                                canvas = np.zeros((h_shape, w_shape), dtype=np.uint8)
                                cv2.fillPoly(canvas, [mask_xy_int], 255)
                                clean_polys = post_processor.process(canvas)
                                
                                if idx not in proc_polygons_by_class:
                                    proc_polygons_by_class[idx] = []
                                proc_polygons_by_class[idx].extend(clean_polys)
                
                # Draw Hollow mapped locally onto exact original size axis
                hollow_drawn = visualizer.draw_hollow(original_bgr, proc_polygons_by_class)
                
                ax.imshow(hollow_drawn)
                ax.set_title(f"{cfg['name']} ({'SAHI Mode' if cfg['use_sahi'] else 'Standard'})", fontsize=14, pad=10)
                ax.axis("off")
            except Exception as e:
                print(f"❌ Render failed for {cfg['name']}: {e}")
                ax.set_title(f"{cfg['name']} (Render Failed)")
                ax.axis("off")
            
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=[c/255 for c in color], edgecolor='k', label=visualizer.class_names.get(id, f"ID {id}")) 
            for id, color in visualizer.class_colors.items()
        ]
        # Attach legend to the last axis outside the rendering boundary
        axes[-1].legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=12)
            
        plt.tight_layout(pad=0.2)
        
        import datetime
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = f"multi_comparison_{timestamp}.png"
        fig.savefig(save_path, dpi=150, bbox_inches='tight', pad_inches=0)
        display(fig)
        plt.close(fig)
        
        print(f"✅ Comparison generation complete!")
        print(f"📷 Image permanently saved to disk as: {save_path} (for GitHub viewing)")

visualize_btn.on_click(run_multi_visualizer)
display(widgets.HTML("<b>Upload a sample to compare across configured models:</b>"))

# Fix UI Spacing: Use GridBox or slim margins to eliminate extra vertical spacing between row 1 and row 2 of UI.
ui_row1 = widgets.HBox([upload_compare_btn, conf_slider], layout=widgets.Layout(margin='0 0 5px 0'))
ui_row2 = widgets.HBox([min_area_slider, epsilon_slider], layout=widgets.Layout(margin='0 0 10px 0'))
ui_container = widgets.VBox([ui_row1, ui_row2, visualize_btn], layout=widgets.Layout(margin='0px', padding='0px'))

display(ui_container)
display(compare_output)

HTML(value='<b>Upload a sample to compare across configured models:</b>')

Output()